## LLama, Mixtral

In [54]:
import pandas as pd

autom_annotated = pd.read_csv("27_01_llama3.csv")

In [55]:
import json

with open("CQA/data/evaluation_tables/dataset_gpt4_evaluation_fewshot.json") as f:
    dataset_gpt4 = json.load(f)

filenames = []
for i in dataset_gpt4:
    filenames.append(i['model_gen'])

autom_annotated['model_gen'] = filenames

In [56]:
import numpy as np
import json
import re

p = re.compile(r'\d+: (\d+)')

mixtral_sums = []
gpt4_sums = []
gpt3_sums = []
perplexity_sums = []
cam_sums = []
human_sums = []
yahoo_sums = []
llama70b_sums = []
llama8b_sums = []


for j in autom_annotated.values.tolist():
    # if sum([int(i) for i in p.findall(j[-3])]) > 19:
    #     print(j)
    if "mixtral" in j[-1]:
        mixtral_sums.append(sum([int(i) for i in p.findall(j[-4])]))
    elif "gpt4" in j[-1]:
        gpt4_sums.append(sum([int(i) for i in p.findall(j[-4])]))
    elif "gpt3" in j[-1]:
        gpt3_sums.append(sum([int(i) for i in p.findall(j[-4])]))
    elif "perplexity" in j[-1]:
        perplexity_sums.append(sum([int(i) for i in p.findall(j[-4])]))
    elif "cam" in j[-1]:
        cam_sums.append(sum([int(i) for i in p.findall(j[-4])]))
    elif "llama3-70b" in j[-1]:
        llama70b_sums.append(sum([int(i) for i in p.findall(j[-4])]))
    elif "llama3-8b" in j[-1]:
        llama8b_sums.append(sum([int(i) for i in p.findall(j[-4])]))
    elif "human" in j[-1]:
        human_sums.append(sum([int(i) for i in p.findall(j[-4])]))
    elif "yahoo" in j[-1]:
        yahoo_sums.append(sum([int(i) for i in p.findall(j[-4])]))
    else:
        print(j['filename'])

print(np.mean(mixtral_sums), np.mean(gpt4_sums), np.mean(gpt3_sums), np.mean(perplexity_sums), \
      np.mean(yahoo_sums), np.mean(human_sums), np.mean(cam_sums), np.mean(llama8b_sums), np.mean(llama70b_sums))
print(np.max(mixtral_sums), np.max(gpt4_sums), np.max(gpt3_sums), np.max(perplexity_sums), \
      np.max(yahoo_sums), np.max(human_sums), np.max(cam_sums), np.max(llama8b_sums), np.max(llama70b_sums))
print(np.median(mixtral_sums), np.median(gpt4_sums), np.median(gpt3_sums), np.median(perplexity_sums), \
      np.median(yahoo_sums), np.median(human_sums), np.median(cam_sums), np.median(llama8b_sums), np.median(llama70b_sums))

16.285 16.731666666666666 15.14 16.47 5.214285714285714 14.125 6.54 15.585 15.74
19 48 45 47 18 19 15 24 20
17.0 17.0 16.0 17.0 3.0 14.0 6.0 17.0 17.0


In [57]:
autom_annotated = autom_annotated.loc[autom_annotated.id_human.notna()]

In [58]:
import json


with open("annotation_pairs_human_397_no_filename.json", 'r') as f:
    human_data = json.load(f)

id2human_annotations = dict(enumerate([list(i.values())[5:-1] for i in human_data]))
id2human_annotations = dict([(i, [[m for m in k if m is not None] for k in j]) for i,j in id2human_annotations.items()])
id2human_annotations = dict([(str(i), j) for i,j in id2human_annotations.items() if None not in j])

In [59]:
import re

p = re.compile(r'\d+: (\d+)')

llm_annotated = dict([(j,[int(m) for m in p.findall(i)]) for i, j in autom_annotated[['evaluation', 'id_human']].values.tolist()])

In [60]:
id2human_annotations_sum = {i:sum([m[0] for m in j if m]) for i, j in id2human_annotations.items()}

In [61]:
id2human_annotations_sum = {i:j for i, j in id2human_annotations_sum.items() if j>0}

In [62]:
llm_annotated_sum = {i:sum(j) for i, j in llm_annotated.items()}

In [63]:
import scipy

x = []
y = []

for i, j in id2human_annotations_sum.items():
    try:
        x.append(llm_annotated_sum[str(int(i)+1)])
        y.append(j)
    except Exception as e:
        pass

In [64]:
scipy.stats.pearsonr(x, y) #llama 3 8b

PearsonRResult(statistic=0.4600948859043356, pvalue=0.00013072829172617155)

In [65]:
import krippendorff

krippendorff.alpha(reliability_data=[x, y])

0.40948677540386125

In [66]:
x_ = []
y_ = []

for i, j in id2human_annotations.items():
    try:
        x_.extend(llm_annotated[str(int(i)+1)])
        y_.extend(j)
    except Exception as e:
        pass

In [67]:
xy = [(n, k[0]) for n, k in zip(x_, y_) if k]
x_ = [x[0] for x in xy]
y_ = [x[1] for x in xy]

In [68]:
scipy.stats.pearsonr(x_, y_) 

PearsonRResult(statistic=0.488237816448539, pvalue=1.3619027992738844e-58)

In [69]:
import krippendorff

krippendorff.alpha(reliability_data=[x_, y_])

0.4817795598442248

## Perplexity, GPT-4, GPT-3.5

In [38]:
path = 'CQA/anonynous_repo/data/evaluated_summaries/gpt-4'

In [39]:
import os
c = 0

gpt35 = []

id_model_106_found = pd.read_csv("id_model_106_found.json")

for x,y,z in os.walk(path):
    
    for i in z:
        with open(os.path.join(x,i)) as f:
            for line in json.load(f):
                filtered_df = id_model_106_found[(id_model_106_found["object1"] == line['object1']) & (id_model_106_found["object2"] == line['object2']) & (id_model_106_found["model_gen"] == i.split(".json")[0])]
                if len(filtered_df) == 1 and filtered_df['comparison'].tolist()[0].split()[:300] == line['comparison'].split()[:300]:
                    line['id_human'] = filtered_df['id_human'].values.tolist()[0]
                    gpt35.append(line)
                    c+=1

In [40]:
import re

with open("xxx.json", 'w') as w:
    json.dump(gpt35, w)
df_gpt35 = pd.read_json("xxx.json")

p = re.compile(r'\d+: (\d+)')

llm_annotated = dict([(j,i) for i, j in df_gpt35[['score_dict', 'id_human']].values.tolist()])
llm_annotated_sum = {i:sum(j.values()) for i, j in llm_annotated.items()}

In [48]:
x = []
y = []

for i, j in id2human_annotations_sum.items():
    try:
        x.append(llm_annotated_sum[int(i)+1])
        y.append(j)
    except Exception as e:
        pass

scipy.stats.pearsonr(x, y) 

PearsonRResult(statistic=0.5529168092144717, pvalue=5.595479772631105e-06)

In [50]:
import krippendorff

krippendorff.alpha(reliability_data=[x, y])

0.39884204722041205

In [51]:
x = []
y = []

for i, j in id2human_annotations.items():
    try:
        assert len(list(llm_annotated[int(i)+1].values())) == len([max(m) for m in j])
        x.extend(list(llm_annotated[int(i)+1].values()))
        y.extend([max(m) for m in j])
    except Exception as e:
        pass

scipy.stats.pearsonr(x, y) 

PearsonRResult(statistic=0.692585106107754, pvalue=2.8515060805668103e-125)

In [52]:
import krippendorff

krippendorff.alpha(reliability_data=[x, y])

0.6904985109869778

In [43]:
import ast
from tqdm import tqdm
import os
import json

c = 0
gpt4_identified = []

annotation_pairs = pd.read_json("annotation_pairs_human_397_no_filename.json")

for x,y,z in os.walk(path):
    
    for i in z:
        with open(os.path.join(x,i)) as f:
            for line in json.load(f):
                line['filename'] = i
                line['Unnamed: 0'] = c
                gpt4_identified.append(line)
                c+=1
id2info = {}
xs = {}
ff = 0
for x, j in enumerate(gpt4_identified):
    for n, i in enumerate(annotation_pairs.values.tolist()):
        inters = set(j['comparison'].split()).intersection(set(i[4].split()))
        if len(inters) == len(set(j['comparison'].split())) and j['object1']==i[1] and j['object2']==i[2]:
            id2info[n] = gpt4_identified[n]
            xs[n] = i
            ff+=1

In [44]:
import numpy as np
mixtral_sums = []
gpt4_sums = []
gpt3_sums = []
perplexity_sums = []
cam_sums = []
human_sums = []
yahoo_sums = []
llama70b_sums = []
llama8b_sums = []


for x, j in enumerate(gpt4_identified):
    if "mixtral" in j['filename']:
        mixtral_sums.append(sum(j['score_dict'].values()))
    elif "gpt4" in j['filename']:
        gpt4_sums.append(sum(j['score_dict'].values()))
    elif "gpt3" in j['filename']:
        gpt3_sums.append(sum(j['score_dict'].values()))
    elif "perplexity" in j['filename']:
        perplexity_sums.append(sum(j['score_dict'].values()))
    elif "cam" in j['filename']:
        cam_sums.append(sum(j['score_dict'].values()))
    elif "llama3-70b" in j['filename']:
        llama70b_sums.append(sum(j['score_dict'].values()))
    elif "llama3-8b" in j['filename']:
        llama8b_sums.append(sum(j['score_dict'].values()))
    elif "human" in j['filename']:
        human_sums.append(sum(j['score_dict'].values()))
    elif "yahoo" in j['filename']:
        yahoo_sums.append(sum(j['score_dict'].values()))
    else:
        print(j['filename'])

print(np.mean(mixtral_sums), np.mean(gpt4_sums), np.mean(gpt3_sums), np.mean(perplexity_sums), \
      np.mean(yahoo_sums), np.mean(human_sums), np.mean(cam_sums), np.mean(llama8b_sums), np.mean(llama70b_sums))
print(np.max(mixtral_sums), np.max(gpt4_sums), np.max(gpt3_sums), np.max(perplexity_sums), \
      np.max(yahoo_sums), np.max(human_sums), np.max(cam_sums), np.max(llama8b_sums), np.max(llama70b_sums))
print(np.median(mixtral_sums), np.median(gpt4_sums), np.median(gpt3_sums), np.median(perplexity_sums), \
      np.median(yahoo_sums), np.median(human_sums), np.median(cam_sums), np.median(llama8b_sums), np.median(llama70b_sums))

17.2 17.765 16.658333333333335 17.415 9.321428571428571 17.0875 9.52 16.58 17.12
19 19 19 19 18 19 17 19 19
17.0 18.0 17.0 18.0 8.5 17.0 9.0 17.0 17.0
